# Machine Learning Project Title
##### *Author: Slavena Peneva-Kargiou*  
##### *Course: SoftUni Machine Learning* 
##### *Instructor: Yordan Darakchiev*
##### *Date: June 2026*

In [2]:
import subprocess
subprocess.run(['pip', 'install', 'jieba'], check=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import spearmanr, mannwhitneyu
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, cross_val_score,
                                     GridSearchCV, KFold)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, classification_report)
from sklearn.decomposition import PCA
from collections import Counter
import jieba
import re
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

# 1. Introduction

## 1.1 Motivation and Context
Children learning their first language acquire words in a consistent and well-documented order. Concrete nouns and social words come first - the things they can see, touch and point at. Pronouns, connectors and quantifiers are among the most common words in child-directed speech, yet children learn them last precisely because they refer to nothing in the physical world.

Language models learn differently. Their only signal is statistical: a word that appears frequently in many different contexts becomes well represented. There is no embodied grounding and no emotional attachment to the sound of a word before its meaning arrives.

In an earlier project, I examined this gap using English child vocabulary data from the Wordbank database and the BabyLM corpus - a curated collection of child-directed English text used to train language models. The finding was clear - corpus frequency predicted almost nothing about the order in which children acquire words, while concreteness was a strong predictor.

This project begins where thatF finding left off and was also inspired by personal experience. I learned Mandarin as an adult, deliberately, through structured study and largely through text. There was no pointing at things and no emotional connection to the language and its speakers. That experience feels structurally different from how children learn their first language and, in some ways, more similar to what a language model does. Both an adult L2 learner and a language model rely heavily on text and explicit pattern recognition. Neither acquires the language through the embodied emotionally-driven process that characterizes child L1 acquisition.

This project follows that intuition with data and ML models using Mandarin as the common language across all three learners and tries to answer the following question: Do the statistical features that predict word acquisition order differ between children learning Mandarin as L1 and adults learning it as L2, and does the adult L2 pattern more closely resemble a language model's frequency-driven learning signal?

## 1.3 The Three Learners
All three learners in this project are working on the same language - Mandarin. Apart from my personal experience learning it as an adult, it is also used for practical and conceptual reasons. Practically, all required datasets are available in Mandarin. Conceptually, it is typologically distant from the Indo-European languages, making it an interesting case.
### 1.3.1 Children
I am using Wordbank Mandarin (Beijing) CDI data. Acquisition order is measured as age of acquisition (AoA) - the youngest age in months at which at least 50% of children produce a given word.
### 1.3.2 Adult L2 learners
HSK level is the vocabulary grading from China's official Mandarin proficiency framework, the Hanyu Shuiping Kaoshi. I use the old HSK system (6 levels, approximately 5000 words), whereby HSK level 1 contains the 150 most basic words, while HSK level 6 is the most advanced. Important limitation stated upfront: the HSK level reflects curriculum design decisions based on expert judgement about what adult learners need to know, not on empirical measurement of what they actually learn first.
### 1.3.3 Language model
It is represented by word frequency in the Chinese BabyLM corpus, a 100-million-token Mandarin corpus designed for the Chinese BabyLM Challenge and built on the same developmental plausibility features as the English BabyLM corpus. Frequency is the model's primary learning signal.

## 1.4 Positioning Statement
The BabyLM challenge (Warstadt et al., 2023 onwards) and its Chinese counterpart (Chinese BabyLM Challenge, Hu et al.) train language models on developmentally plausible data, using child acquisition as the benchmark for human-like language learning. This project does not enter either challenge but engages with their central premise, and furthermore asks whether adult L2 acquisition might be a natural complement or alternative reference point to child acquisition when thinking about how models learn language.

No such three-way comparison - children, adult L2 learners and language models, on the same language, appears to exist in the published literature. This is an exploratory study. The datasets are real, the methods are careful and the conclusions are stated honestly. Every limitation is acknowledged explicitly.

## 1.5 Theoretical Framework and Hypotheses
The three learners in this project occupy different positions on a spectrum defined by how much they rely on embodied social experience versus statistical patterns in text when acquiring language.

Children learning L1 are at the fully embodied end. Their early vocabulary is shaped by what they can perceve and interact, which is why concreteness drives acquisition order.

Adult L2 learners occupy a middle position. They bring full conceptual knowledge of the world to the new language, but their learning is primarily text driven and explicit.

Language models are at the fully statistical end, frequency being their only signal.

This suggests three testable predictions:

$
H_1: \text{Frequency is a stronger predictor of adult L2 acquisition order (HSK level) than of child L1 acquisition order (AoA).}
$

$
H_2: \text{Concreteness is a weaker predictor of adult L2 acquisition order than of child L1 acquisition order.}
$

$
H_3: \text{The overall feature importance profile of the adult L2 model resembles the model's frequency-based learning signal more closely than the child model does.}
$

These are exploratory predictions, not formal null hypotheses. The project follows them with data and reports what it finds.

# 2. The Datasets
Four datasets provide the raw material for this project. Two provide the target variables - AoA for children (Wordbank) and HSK level for adult L2 learners. One provides the model's learning signal - word frequency from the Chinese BabyLM corpus. One provides concreteness ratings - initially Xu & Li (2020), later suplemented by Liu et al. (2007) for reasons explained below. Word length in characters is derived directly from the word strings and requires no external source.

The datasets are merged into two parallel analysis datasets:
- Child dataset: Wordbank words with AoA and all available features
- Adult dataset: HSK words with HSK level and all available features


## 2.1 Wordbank Mandarin
Wordbank (wordbank.stanford.edu) is the largest open repository of children's vocabulary development data, aggregating responses from the MacArthur-Bates Communicative Development inventory (CDI) across 42 languages. The Beijing Mandarin WS form is used here, covering children aged 16-30 month. Each word in the dataset has a production proportion for each age in month - the fraction of children reported to produce that word at that age. AoA is defined as the younget age at which this proportion first reaches or exceeds 50%.

The Wordbank data uses Chinese characters throughout. Some entries contain parenthetical notes or punctuation that do not appear in any other dataset, for example 旺旺（狗叫）meaning "woof woof (dog sound)". These are stripped before merging. 

In [13]:
df_wordbank = pd.read_csv('wordbank_mandarin_items.csv',
                           encoding='utf-8', quoting=3)
df_wordbank.columns = [c.strip('"') for c in df_wordbank.columns]
df_wordbank['item_definition'] = df_wordbank['item_definition'].str.strip('"')
df_wordbank['category'] = df_wordbank['category'].str.strip('"')

# Strip parenthetical disambiguators e.g. 旺旺（狗叫）→ 旺旺
# These appear in Wordbank but not in any other dataset
df_wordbank['word'] = df_wordbank['item_definition']\
    .str.replace(r'（[^）]*）', '', regex=True)\
    .str.replace(r'\([^)]*\)', '', regex=True)\
    .str.replace(r'[？！。，、]', '', regex=True)\
    .str.strip()

# Further cleaning — strip spaces, take first slash-separated option,
# remove trailing punctuation
df_wordbank['word'] = df_wordbank['word']\
    .str.replace(' ', '', regex=False)\
    .apply(lambda w: w.split('/')[0] if '/' in w else w)\
    .str.replace(r'[！!？?。，、]', '', regex=True)\
    .str.strip()

age_cols = [str(a) for a in range(16, 31)]

def compute_aoa(row, threshold=0.5):
    """Return the first age (months) at which >= 50% of children produce the word."""
    for age in age_cols:
        try:
            if float(row[age]) >= threshold:
                return int(age)
        except:
            pass
    return np.nan

df_wordbank['aoa'] = df_wordbank.apply(compute_aoa, axis=1)
df_aoa = df_wordbank[['word', 'category', 'aoa']].copy()

print(f"Total Wordbank words:             {len(df_aoa)}")
print(f"Words with AoA defined (≥50%):   {df_aoa['aoa'].notna().sum()}")
print(f"Words never reaching threshold:  {df_aoa['aoa'].isna().sum()}")
display(df_aoa.head(5))

Total Wordbank words:             799
Words with AoA defined (≥50%):   772
Words never reaching threshold:  27


,word,category,aoa
0,喂,sounds,16.0
1,旺旺,sounds,16.0
2,喵,sounds,16.0
3,嘀嘀,sounds,17.0
4,哎哟,sounds,17.0


## 2.2 HSK Vocabulary Lists
The old HSK system (6 levels, approximately 5000 words) is used. The words appearing in multiple levels are resolved by keeping the lowest level - since level 1 means learned first, this seems to be the most logical and methodologically appropriate choice.

In [4]:
hsk_dfs = []
for level in range(1, 7):
    df = pd.read_csv(f'hsk{level}.csv',
                     header=None, names=['word', 'pinyin', 'english'])
    df['hsk_level'] = level
    hsk_dfs.append(df)

df_hsk = pd.concat(hsk_dfs, ignore_index=True)

# Resolve duplicates — keep lowest level
df_hsk = df_hsk.sort_values('hsk_level').drop_duplicates(
    subset='word', keep='first').reset_index(drop=True)

print(f"Total HSK words after deduplication: {len(df_hsk)}")
print(f"\nWords per level:")
print(df_hsk['hsk_level'].value_counts().sort_index())
display(df_hsk.head(5))

Total HSK words after deduplication: 4993

Words per level:
hsk_level
1     150
2     147
3     298
4     598
5    1300
6    2500
Name: count, dtype: int64


,word,pinyin,english,hsk_level
0,开,kāi,to open,1
1,看,kàn,to see,1
2,看见,kàn jiàn,to see,1
3,块,kuài,lump (of earth),1
4,来,lái,to come,1


## 2.3 Chinese BabyLM Corpus - Computing Word Frequencies
The Chinese BabyLM corpus is a 100-million-token Mandarin dataset built from child-directed speech, children's books, educational text and subtitles, designed for the Chinese BabyLM Challenge. It is the Mandarin equivalent of the English BabyLM corpus and provides the model's learning signal throughout this project.

Word frequencies are computed using jieba, the standard Chinese word segmentation library, which splits unsegmented Chinese text into words. Because this computation takes some time on the full corpus, the results are saved to a csv file. This precomputed file is included in the project repository so the notebook can be run without repeating the computation.

In [5]:
FREQ_CACHE = 'babylm_zh_frequencies.csv'

if os.path.exists(FREQ_CACHE):
    # Load pre-computed frequencies — no need to reprocess
    df_freq = pd.read_csv(FREQ_CACHE)
    print(f"Loaded pre-computed frequencies: {len(df_freq):,} unique words")
else:
    # Compute from scratch — only needed once
    print("Computing frequencies from Chinese BabyLM corpus...")
    df_babylm = pd.read_parquet('train-00000-of-00001.parquet')
    print(f"Corpus: {len(df_babylm):,} documents")
    print(f"Categories: {df_babylm['category'].value_counts().to_dict()}")

    word_counts = Counter()
    for i, text in enumerate(df_babylm['text']):
        words = jieba.lcut(str(text))
        word_counts.update(words)
        if (i + 1) % 10000 == 0:
            print(f"  Processed {i+1:,} / {len(df_babylm):,} documents...")

    df_freq = pd.DataFrame({
        'word':      list(word_counts.keys()),
        'frequency': list(word_counts.values())
    })

    # Remove punctuation, whitespace and non-Chinese tokens
    # \u4e00-\u9fff is the Unicode range covering all standard Chinese characters
    chinese_pattern = re.compile(r'[\u4e00-\u9fff]')
    df_freq = df_freq[df_freq['word'].apply(
        lambda w: bool(chinese_pattern.search(str(w)))
    )].copy().reset_index(drop=True)
    print(f"After cleaning: {len(df_freq):,} unique Chinese words")

    # Save for future use
    df_freq.to_csv(FREQ_CACHE, index=False, encoding='utf-8-sig')
    print(f"Saved to {FREQ_CACHE}")

print(f"\nFrequency range: {df_freq['frequency'].min():,} – "
      f"{df_freq['frequency'].max():,}")
print(f"Total tokens: {df_freq['frequency'].sum():,}")
display(df_freq.sort_values('frequency', ascending=False).head(10))

Loaded pre-computed frequencies: 606,029 unique words

Frequency range: 1 – 3,898,022
Total tokens: 75,761,588


,word,frequency
5,的,3898022
238,我,2415696
25,了,2255739
241,你,2078294
158,是,1403442
248,啊,842693
39,在,767953
105,就,745045
22,他,727203
120,不,574433


## 2.4 Xu & Li Concreteness Norms
Xu & Li (2020) provide concreteness ratins for 9877 two-character Chinese words collected from native Mandarin speakers. 

One important preprocessing step: the Xu & Li scale runs from 1 (very concrete) to 5 (very abstract) - the inverse of standard direction used in most English-language research. To ensure consistency with my earlier project, scores are reversed before the analysis:
$$\text{concreteness} = 6 - \text{concreteness}_{\text{original}}$$
After reversal, a score of 5 indicates maximum concreteness and a score of 1 indicates maximum abstractness.

In [6]:
df_xuli = pd.read_excel(
    'Concretenss_Ratings_of_9877_Two_Character_Chinese_Words.xlsx')
df_xuli = df_xuli.rename(columns={
    'Word':                  'word',
    'Mean of Valid Ratings': 'concreteness_raw'
})
df_xuli['word'] = df_xuli['word'].str.strip()

# Reverse scale: original 1=concrete, 5=abstract → reversed 5=concrete, 1=abstract
df_xuli['concreteness'] = 6 - df_xuli['concreteness_raw']

print(f"Xu & Li norms: {len(df_xuli):,} words")
print(f"Reversed scale range: {df_xuli['concreteness'].min():.2f} – "
      f"{df_xuli['concreteness'].max():.2f}  (1=abstract, 5=concrete)")
display(df_xuli[['word', 'concreteness_raw', 'concreteness']].head(5))

Xu & Li norms: 9,877 words
Reversed scale range: 1.44 – 4.96  (1=abstract, 5=concrete)


,word,concreteness_raw,concreteness
0,暗笑,2.666667,3.333333
1,黯然,4.185185,1.814815
2,昂首,2.222222,3.777778
3,白人,1.629630,4.370370
4,绑架,1.777778,4.222222


To assess coverage I performed a preliminary merge before commiting to this dataset as the sole concreteness source:

In [7]:
df_freq['log_frequency'] = np.log10(df_freq['frequency'].clip(lower=1))

def preliminary_merge(df_target, target_col):
    """Quick merge to check feature coverage."""
    df = df_target[['word', target_col]].copy()
    df = df.merge(df_freq[['word', 'log_frequency']], on='word', how='left')
    df = df.merge(df_xuli[['word', 'concreteness']], on='word', how='left')
    df['word_length'] = df['word'].str.len()
    return df

df_child_prelim = preliminary_merge(
    df_aoa.dropna(subset=['aoa']), 'aoa')
df_adult_prelim = preliminary_merge(df_hsk, 'hsk_level')

print("Coverage with Xu & Li alone:")
print(f"  Child:  {df_child_prelim[['aoa','log_frequency','concreteness']].notna().all(axis=1).sum()} / {len(df_child_prelim)} words")
print(f"  Adult:  {df_adult_prelim[['hsk_level','log_frequency','concreteness']].notna().all(axis=1).sum()} / {len(df_adult_prelim)} words")

Coverage with Xu & Li alone:
  Child:  180 / 772 words
  Adult:  2757 / 4993 words


The results here are rather disappointing. Only 180 child words with all features present out of 772 is obviously too small a basis for ML models. The problem is concreteness - the Xu & Li dataset exclusively covers two-character wwords. However, early child vocabulary is dominated by single-character words like 我 (I), 你 (you), 水 (water), 狗 (dog), as well as reduplicated words where the same character is doubled, like 妈妈 (mom), 爸爸 (dad), 姐姐 (older sister), 宝宝 (baby). These are among the most concrete and earliest acquired words in Mandarin, yet they are entirely absent from the Xu & Li norms. This is something that deffinitely needed to be addressed, yet, as it appeared, no single freely available concreteness dataset covers all these types of words. The logical solution was to find complementary dataset.

## 2.5 Liu et al. (2007) Concreteness Norms - Single Characters
Liu et al. (2007) provide psycholinguistic norms including concretenenss for 2390 simplified Mandarin single-character words, collected from 480 native Chinese speakers. The database is freely accessible online at the Chinese Single-character Word Database (CSWD). The concreteness scale runs from 0 to 7 where higher scores indicate greater concreteness. This coincides with the direction I am using throughout the project, so no reversal is needed.

To combine Liu et al. with Xu & Li, both datasets are normalized to a common 0-1 scale using min-max normalization:

$$
\text{concreteness}_{\text{norm}} =
\frac{\text{score} - \text{min}}{\text{max} - \text{min}}
$$

The two datasetd cover non-overlapping vocabulary - single-character and two-character resepctively. Therefore, there is no need for deduplication and they are siply concatenated after normalization. 


In [8]:
df_liu = pd.read_csv('liu_2007_single_char.txt', sep='\t',
                      encoding='utf-8')
df_liu = df_liu.rename(columns={
    'Word': 'word',
    'CON':  'concreteness_raw'  # adjust to actual column name
})
df_liu['word'] = df_liu['word'].str.strip()

print(f"Liu et al. norms: {len(df_liu):,} single-character words")
print(f"Scale range: {df_liu['concreteness_raw'].min():.2f} – "
      f"{df_liu['concreteness_raw'].max():.2f}  (higher = more concrete)")

# Verify direction — most concrete words should be tangible objects
print("\nMost concrete words (highest scores):")
display(df_liu.nlargest(5, 'concreteness_raw')[['word', 'concreteness_raw']])
print("\nMost abstract words (lowest scores):")
display(df_liu.nsmallest(5, 'concreteness_raw')[['word', 'concreteness_raw']])

Liu et al. norms: 2,356 single-character words
Scale range: 1.90 – 7.00  (higher = more concrete)

Most concrete words (highest scores):


,word,concreteness_raw
1012,梨,7.00
1132,猫,7.00
1583,鼠,7.00
1673,桃,7.00
1526,绳,6.97



Most abstract words (lowest scores):


,word,concreteness_raw
737,即,1.90
1856,现,2.20
462,菲,2.23
2220,臻,2.25
515,该,2.35


In [9]:
def normalize_concreteness(series):
    """Min-max normalize concreteness scores to 0-1 range."""
    return (series - series.min()) / (series.max() - series.min())

df_xuli['concreteness_norm'] = normalize_concreteness(df_xuli['concreteness'])
df_liu['concreteness_norm'] = normalize_concreteness(df_liu['concreteness_raw'])

# Concatenate directly — no overlap between single-character and two-character datasets
df_concrete_combined = pd.concat([
    df_xuli[['word', 'concreteness_norm']],
    df_liu[['word', 'concreteness_norm']]
], ignore_index=True)

print(f"Xu & Li coverage:        {len(df_xuli):,} words")
print(f"Liu et al. coverage:     {len(df_liu):,} words")
print(f"Combined total:          {len(df_concrete_combined):,} words")

Xu & Li coverage:        9,877 words
Liu et al. coverage:     2,356 words
Combined total:          12,233 words


## 2.6 Handling Reduplicated Words
Mandarin contains a class of words formed by doubling a single character, for example 爸爸 (dad), 弟弟 (younger brother), etc. These appear frequently in Wordbank but are absent from both concreteness datasets because they are two-character words derived from single characters.

Their concreteness is derived from the base character's rating in the Liu et a. dataset, because linguistically reduplicated words derive their meaning entirely from the base character.

In [10]:
def is_reduplication(word):
    """Return True if word is a two-character reduplication (e.g. 妈妈)."""
    return len(word) == 2 and word[0] == word[1]

# Build lookup from Liu et al. normalized scores
liu_lookup = dict(zip(df_liu['word'], df_liu['concreteness_norm']))

# Find reduplicated Wordbank words still missing concreteness after main merge
missing_conc_words = df_aoa[
    ~df_aoa['word'].isin(df_concrete_combined['word'])
]['word'].tolist()

reduplications = [w for w in missing_conc_words if is_reduplication(w)]
print(f"Reduplicated words missing concreteness: {len(reduplications)}")
print(reduplications)

# Create supplementary concreteness entries for reduplications
reduplication_rows = []
for word in reduplications:
    base_char = word[0]
    if base_char in liu_lookup:
        reduplication_rows.append({
            'word': word,
            'concreteness_norm': liu_lookup[base_char]
        })

df_reduplications = pd.DataFrame(reduplication_rows)
print(f"\nReduplications with base character in Liu et al.: "
      f"{len(df_reduplications)}")

# Add to combined concreteness dataset
df_concrete_final = pd.concat([
    df_concrete_combined,
    df_reduplications
], ignore_index=True).drop_duplicates(subset='word', keep='first')

print(f"\nFinal concreteness dataset: {len(df_concrete_final):,} unique words")

Reduplicated words missing concreteness: 25
['旺旺', '嘀嘀', '咩咩', '嘎嘎', '梆梆', '喳喳', '爸爸', '奶奶', '爷爷', '宝宝', '姑姑', '叔叔', '伯伯', '舅舅', '姐姐', '妹妹', '哥哥', '弟弟', '等等', '画画', '数数', '掸掸', '猩猩', '蛐蛐', '星星']

Reduplications with base character in Liu et al.: 17

Final concreteness dataset: 12,250 unique words


## 2.7 Building the Two Parallel Datasets
With the combined concreteness source in place, the two analysis datasets are built using the same merge structure.

In [16]:
def build_dataset(df_target, target_col, word_col='word'):
    """
    Merge a target dataset (Wordbank or HSK) with all feature sources.
    Left join on target words — all target words retained,
    features added where available.
    """
    df = df_target[[word_col, target_col]].copy()
    if 'category' in df_target.columns:
        df['category'] = df_target['category']

    # Frequency from Chinese BabyLM corpus
    df = df.merge(df_freq[['word', 'frequency', 'log_frequency']],
                  on='word', how='left')

    # Concreteness — combined from Xu & Li, Liu et al., and reduplications
    df = df.merge(df_concrete_final[['word', 'concreteness_norm']],
                  on='word', how='left')
    df = df.rename(columns={'concreteness_norm': 'concreteness'})

    # Word length in characters — derived directly
    df['word_length'] = df['word'].str.len()

    return df

df_child = build_dataset(df_aoa.dropna(subset=['aoa']), target_col='aoa')
df_adult = build_dataset(df_hsk, target_col='hsk_level')

FEATURES = ['log_frequency', 'concreteness', 'word_length']

for name, df, target in [('Child (Wordbank)', df_child, 'aoa'),
                           ('Adult (HSK)',     df_adult, 'hsk_level')]:
    print(f"=== {name} ===")
    print(f"Total words:        {len(df)}")
    print(f"With frequency:     {df['frequency'].notna().sum()}")
    print(f"With concreteness:  {df['concreteness'].notna().sum()}")
    print(f"With all features:  "
          f"{df[[target]+FEATURES].notna().all(axis=1).sum()}")
    print()

# Final analysis datasets — words with all features present
df_child_analysis = df_child.dropna(
    subset=['aoa'] + FEATURES).copy().reset_index(drop=True)
df_adult_analysis = df_adult.dropna(
    subset=['hsk_level'] + FEATURES).copy().reset_index(drop=True)

print(f"Final child analysis dataset:  {len(df_child_analysis)} words")
print(f"Final adult analysis dataset:  {len(df_adult_analysis)} words")

=== Child (Wordbank) ===
Total words:        772
With frequency:     743
With concreteness:  505
With all features:  504

=== Adult (HSK) ===
Total words:        4993
With frequency:     4983
With concreteness:  3269
With all features:  3269

Final child analysis dataset:  504 words
Final adult analysis dataset:  3269 words


The two analysis datasets differ substantially in size - 504 child words versus 3269 adult words. This reflects both the smaller scope of the Wordbank checklist reative to the HSK vocabulary lists, as well as the differential coverage of available concretenessnorms accross word types.

## 2.8 What Gets Lost in the Merge and Why

In [17]:
missing_child_freq = df_child[
    df_child['frequency'].isna()]['word'].tolist()
missing_child_conc = df_child[
    df_child['concreteness'].isna() &
    df_child['frequency'].notna()]['word'].tolist()
missing_adult_freq = df_adult[
    df_adult['frequency'].isna()]['word'].tolist()
missing_adult_conc = df_adult[
    df_adult['concreteness'].isna() &
    df_adult['frequency'].notna()]['word'].tolist()

print(f"Child words missing from corpus ({len(missing_child_freq)}):")
print(missing_child_freq[:15])
print(f"\nChild words missing concreteness ({len(missing_child_conc)}):")
print(missing_child_conc[:15])
print(f"\nHSK words missing from corpus ({len(missing_adult_freq)}):")
print(missing_adult_freq[:15])
print(f"\nHSK words missing concreteness ({len(missing_adult_conc)}):")
print(missing_adult_conc[:15])

Child words missing from corpus (29):
['咩咩', '嘎嘎达', '自己的名字', '小朋友的名字', '藏猫猫', '抓住了', '你拍一我拍一', '好吧', '对了', '亮了', '倒了', '掸掸', '没了', '坏了', '破了']

Child words missing concreteness (239):
['喂', '喵', '嘀嘀', '哎哟', '呀', '梆梆', '喳喳', '嗷', '姥姥', '姥爷', '阿姨', '大妈', '唐老鸭', '米老鼠', '孙悟空']

HSK words missing from corpus (10):
['呢,ne,particle indicating that a previously asked question is to be applied to the preceding word ("What about ...?"', '不客气', '比,bǐ,(particle used for comparison and "-er than")', '分之', '系领带', '要不', '和气', '纽扣儿', '哇,wa,replaces 啊 when following the vowel "u" or "ao";', '无可奉告,wú kě fèng gào,(idiom) "no comment";']

HSK words missing concreteness (1714):
['块', '了', '你', '年', '女儿', '谁', '什么', '十', '时候', '四', '岁', '他', '她', '太', '喂']


The remaining merge losses fall into identifiable and linguistically interpretable categories.

Child words missing from the corpus are represented by certain game chants, phrases and slash-separated alternatives, that exst as single Wordbank items but cannot be matched to corpus tokens. Child words still missing concreteness include sound words whose base characters have no independent meaning, as well as proper names of fictional characters like 唐老鸭 (Donald Duck) and 米老鼠 (Mickey Mouse). The 1714 HSK words missng concreteness include very common words like 你 (you), 他 (he), 了 (aspect marker). This is not a data quality problem - both Liu et al., and Xu & Li deliberately exclude pronouns, particles and functional words, because concreteness as a psycholinguistic construct has no meaningful napplication to purely grammatical words. The missing words are therefore not random: they are systematically the most grammatical end of the HSK vocabulary, and the adult analysis dataset is biased toward content words as a result.

The size difference between the two final datasets - 504 child words versus 3269 adult words, reflects these combined gaps. Absolute model performance metrics such as R² and classification accuracy are not directly comparable between the two models. The analysis focuses on feature importance profiles and Spearman correlations, which are internally valid within each dataset and support meaningful comparisson across them.

# 3. Exploratory Data Analysis

## 3.1 AoA and HSK Level Distributions

## 3.2 Frequency Distributions and Zipf's Law

## 3.3 Concreteness Distributions

## 3.4 Spearman Correlations

## 4.5 Scatter Plots

# 5. Machine Learning Models

## 5.1 Feature Set and Setup

## 5.2 Model 1 - Predicting Child AoA 

## 5.3 Model 2 - Predicting HSK Level

## 5.4 Lasso - Feature Selection ???

# 6. The Three Learners - Feature Importance Comparison

## 6.1 Coefficient Comparison

## 6.2 Feature Importances

## 6.3 PCA

## 6.4 Summary: Where Does the Adult L2 Learner Sit?

# 7. Discussion and Limitations
## 7.1 What the Models Found
## 7.2 Limitations
## 7.3 What a Larger Study Might Look Like

# 8. Conclusion

# 9. References